# XB1_labels_ja (GPU): ja_s0 refusal labels on the extraction twins (copied from steering S1_labels)

In [1]:
WORK = "/content/x2"
MODELS = ["ja_s0"]
OUTNAME = "s1_labels_ja_s0"


In [2]:
# HF_TOKEN: taken from the environment, else from the HF token file on the VM. Never printed or saved.
import os
from pathlib import Path as _P
if not os.environ.get("HF_TOKEN"):
    _tf = _P.home() / ".cache" / "huggingface" / "token"
    if _tf.exists():
        os.environ["HF_TOKEN"] = _tf.read_text().strip()
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not available"


In [3]:
# Shared helper functions (AGENT_PROMPT_EN 0.8) -- identical in every notebook that uses them.
import re as _re

ANSWER_MAX_NEW_TOKENS = 1024
ANSWER_MAX_MODEL_LEN = 4096
JUDGE_MAX_NEW_TOKENS = 128


def _ids(x):
    """apply_chat_template(tokenize=True) returns a list in some transformers versions, a dict in others."""
    if isinstance(x, dict) or hasattr(x, "keys"):
        x = x["input_ids"]
    return [int(t) for t in x]


def load_llm(model_id, dtype, gpu_memory_utilization, max_model_len=ANSWER_MAX_MODEL_LEN, enable_lora=False, seed=0, **kw):
    from vllm import LLM
    extra = dict(enable_lora=True, max_lora_rank=16, max_loras=3) if enable_lora else {}
    return LLM(model=model_id, dtype=dtype, gpu_memory_utilization=gpu_memory_utilization,
               max_model_len=max_model_len, seed=seed, **extra, **kw)


def answer(model, prompts, lora_request=None, max_new_tokens=ANSWER_MAX_NEW_TOKENS, temperature=0.0, top_p=1.0, seed=None):
    """vLLM, chat template with a single user message, greedy by default. Returns text, token count, finish_reason."""
    from vllm import SamplingParams
    from vllm.inputs import TokensPrompt
    tok = model.get_tokenizer()
    reqs = [TokensPrompt(prompt_token_ids=_ids(tok.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=True, add_generation_prompt=True, return_dict=False)))
        for p in prompts]
    sp = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, seed=seed)
    outs = model.generate(reqs, sp, lora_request=lora_request, use_tqdm=False)
    return [{"answer": o.outputs[0].text, "tokens": len(o.outputs[0].token_ids),
             "finish_reason": o.outputs[0].finish_reason} for o in outs]


_SAFETY = _re.compile(r"^Safety:\s*(Safe|Unsafe|Controversial)\s*$", _re.MULTILINE)
_CATS = _re.compile(r"^Categories:\s*(.*)$", _re.MULTILINE)
_REFUSAL = _re.compile(r"^Refusal:\s*(Yes|No)\s*$", _re.MULTILINE)


def parse_judge(raw):
    s, c, r = _SAFETY.search(raw), _CATS.search(raw), _REFUSAL.search(raw)
    label = s.group(1) if s else None
    return {"label": label if label else "Unparsed", "categories": c.group(1).strip() if c else None,
            "refused": (1 if r.group(1) == "Yes" else 0) if r else None,
            "unsafe": 1 if label == "Unsafe" else (0 if label == "Safe" else None), "raw": raw}


def judge(judge_model, prompt, answer):
    """Qwen3Guard response moderation. prompt/answer may be a str or equal-length lists (batched)."""
    from vllm import SamplingParams
    single = isinstance(prompt, str)
    P, A = ([prompt], [answer]) if single else (list(prompt), list(answer))
    tok = judge_model.get_tokenizer()
    texts = [tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}],
                                     tokenize=False, add_generation_prompt=True) for p, a in zip(P, A)]
    outs = judge_model.generate(texts, SamplingParams(temperature=0.0, max_tokens=JUDGE_MAX_NEW_TOKENS), use_tqdm=False)
    res = [parse_judge(o.outputs[0].text) for o in outs]
    return res[0] if single else res


_JA_CHARS = _re.compile(r"[぀-ゟ゠-ヿㇰ-ㇿｦ-ﾟ㐀-䶿一-鿿豈-﫿]")
_LATIN = _re.compile(r"[A-Za-zＡ-Ｚａ-ｚ]")


def answer_lang(text):
    """'ja' if (kana + kanji) / (Latin + kana + kanji) > 0.3, else 'en'."""
    j, l = len(_JA_CHARS.findall(text or "")), len(_LATIN.findall(text or ""))
    return "ja" if (j + l) and j / (j + l) > 0.3 else "en"


# FU deviation 4 (G4 unavailable, run on L4 24 GB): the fractions in these notebooks were sized for the 96 GB G4.
# On a GPU of 80 GB or less they are scaled by 0.88/0.47, so the largest pair loaded together (0.35 + 0.12) fills <= 88%.
_load_llm_g4 = load_llm


def load_llm(model_id, dtype, gpu_memory_utilization, *a, **kw):
    import torch
    tot = torch.cuda.get_device_properties(0).total_memory / 2**30
    u = gpu_memory_utilization if tot > 80 else round(min(0.88, gpu_memory_utilization * 0.88 / 0.47), 3)
    if u != gpu_memory_utilization:
        print(f"load_llm: {model_id} gpu_memory_utilization {gpu_memory_utilization} -> {u} ({tot:.0f} GiB GPU)")
    return _load_llm_g4(model_id, dtype, u, *a, **kw)


In [4]:
# Environment: packages, GPU, dtype, versions. HF_TOKEN comes only from the environment and is never printed.
import importlib.util, subprocess, sys, os, json, hashlib, time, random, gc
from pathlib import Path
need = [p for p, m in [("vllm", "vllm"), ("peft", "peft")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
import torch, numpy as np, pandas as pd, transformers, vllm, peft
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set in the kernel environment"
cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
HARDWARE = "G4" if ("RTX PRO 6000" in GPU_NAME or "Blackwell" in GPU_NAME) else ("T4" if "T4" in GPU_NAME else GPU_NAME)
DTYPE = "bfloat16" if cap[0] >= 8 else "float16"
VERSIONS = {"torch": torch.__version__, "vllm": vllm.__version__, "transformers": transformers.__version__,
            "peft": peft.__version__, "python": sys.version.split()[0]}
print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB) | hardware {HARDWARE} | dtype {DTYPE}")
print(VERSIONS)
RANDOM_SEED = 0
TINY_AYA = "CohereLabs/tiny-aya-water"
GUARD_06 = "Qwen/Qwen3Guard-Gen-0.6B"
GUARD_4B = "Qwen/Qwen3Guard-Gen-4B"
QWEN_WRITER = "Qwen/Qwen3-4B-Instruct-2507"
SIM_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def model_sha(mid):
    from huggingface_hub import HfApi
    return HfApi().model_info(mid).sha


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []


def append_jsonl(p, rows):
    with open(p, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


GPU: NVIDIA L4 (24 GB) | hardware NVIDIA L4 | dtype bfloat16
{'torch': '2.14.0+cu130', 'vllm': '0.33.0', 'transformers': '5.20.2', 'peft': '0.21.2', 'python': '3.13.16'}


In [5]:
# Colab ships torchao 0.10, which PEFT >= 0.21 rejects with an ImportError while dispatching LoRA layers.
# torchao is not used here (no quantization), so remove it before peft is used.
import importlib.metadata as _md
try:
    _tao = _md.version("torchao")
except _md.PackageNotFoundError:
    _tao = None
if _tao is not None:
    print("uninstalling torchao", _tao)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)


In [6]:
# free_llm copied from J1_chunk_en.
def free_llm(llm):
    eng = getattr(llm, "llm_engine", None)
    core = getattr(eng, "engine_core", None)
    if core is not None and hasattr(core, "shutdown"):
        core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()


In [7]:
def ensure_merged(name):
    """Merged bf16 model for adapter `name` (inputs/adapters/<name>), merge_and_unload, saved once on the VM disk (J1 merge code)."""
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    TORCH_DTYPE = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
    ad = IN / "adapters" / name
    mp = Path(WORK) / "merged" / name
    if not (mp / "config.json").exists():
        b = AutoModelForCausalLM.from_pretrained(TINY_AYA, dtype=TORCH_DTYPE, device_map="cpu")
        PeftModel.from_pretrained(b, str(ad)).merge_and_unload().save_pretrained(mp)
        AutoTokenizer.from_pretrained(TINY_AYA).save_pretrained(mp)
        del b
        gc.collect()
    return str(mp)


In [8]:
# Shared steering-experiment helpers (paths, publishing, checkpoints, model sources, weight edit, HF batching).
IN = Path(WORK) / "inputs"
OUT = Path(WORK) / "out"
OUT.mkdir(parents=True, exist_ok=True)
TORCH_DTYPE = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
LAYERS = [8, 12, 16, 18, 20, 24, 28]


def write_jsonl(path, rs):
    with open(path, "w", encoding="utf-8") as f:
        for r in rs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def folder_ok(d):
    d = Path(d)
    if not (d / "DONE").exists() or not (d / "manifest.json").exists():
        return False
    if (d / "DONE").read_text().strip() != sha256_file(d / "manifest.json"):
        return False
    man = json.loads((d / "manifest.json").read_text())
    return all((d / f).exists() and sha256_file(d / f) == m["sha256"] for f, m in man["files"].items())


def publish(d, files, meta):
    """Data files are already written; write manifest.json (counts, SHA-256) then DONE (= sha256 of manifest)."""
    d = Path(d)
    for f in files:
        assert (d / f).exists() and (d / f).stat().st_size > 0, f"missing or empty output {f}"
    man = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "hardware": HARDWARE, "gpu_name": GPU_NAME,
           "dtype": DTYPE, "versions": VERSIONS, **meta,
           "files": {f: {"bytes": (d / f).stat().st_size, "sha256": sha256_file(d / f),
                         "rows": (sum(1 for _ in open(d / f, encoding="utf-8")) if f.endswith(".jsonl") else None)} for f in files}}
    (d / "manifest.json").write_text(json.dumps(man, indent=1, ensure_ascii=False, default=str))
    (d / "DONE").write_text(sha256_file(d / "manifest.json") + "\n")
    return man


plan = json.loads((IN / "plan.json").read_text())
assert (IN / "plan.DONE").read_text().strip() == sha256_file(IN / "plan.json"), "plan.DONE mismatch"
SETS = {k: plan["sets"][k] for k in plan["sets"]}
print({k: len(v) for k, v in SETS.items()})


def model_source(m):
    """'base' -> hub id; otherwise the merged adapter inputs/adapters/<m> on the VM disk (generic ST1 version)."""
    if m == "base":
        return TINY_AYA
    return ensure_merged(m)


def chat_ids(tok, text):
    return _ids(tok.apply_chat_template([{"role": "user", "content": text}], tokenize=True, add_generation_prompt=True, return_dict=False))


def left_pad(seqs, pad_id):
    n = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), n), pad_id, dtype=torch.long)
    att = torch.zeros((len(seqs), n), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, n - len(s):] = torch.tensor(s)
        att[i, n - len(s):] = 1
    return ids, att


def hf_load(path):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(path)
    mod = AutoModelForCausalLM.from_pretrained(path, dtype=TORCH_DTYPE).to("cuda").eval()
    return mod, tok


@torch.no_grad()
def last_token_acts(mod, tok, texts, layers, bs=32):
    """Residual stream at the last prompt token; hidden_states[L+1] is the output of decoder layer L. Returns {L: float32 [n, d]}."""
    seqs = [chat_ids(tok, t) for t in texts]
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
    res = {L: np.zeros((len(seqs), mod.config.hidden_size), dtype=np.float32) for L in layers}
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        ids, att = left_pad([seqs[j] for j in idx], 0)
        pos = (att.cumsum(-1) - 1).clamp(min=0)
        out = mod(input_ids=ids.cuda(), attention_mask=att.cuda(), position_ids=pos.cuda(), output_hidden_states=True, use_cache=False, logits_to_keep=1)
        hs = out.hidden_states
        for L in layers:
            res[L][idx] = hs[L + 1][:, -1, :].float().cpu().numpy()
    return res


def check_writers(mod):
    """Residual-stream writers: every layer's o_proj and down_proj (out x in = hidden x *), no bias."""
    for lyr in mod.model.layers:
        for W in (lyr.self_attn.o_proj, lyr.mlp.down_proj):
            assert W.weight.shape[0] == mod.config.hidden_size and W.bias is None


TIED = None


@torch.no_grad()
def apply_edit(mod, d, lam):
    """W <- W - lam * d (d^T W) for every o_proj / down_proj; E <- E - lam (E d) d^T only if embeddings are not tied."""
    global TIED
    check_writers(mod)
    dv = torch.tensor(np.asarray(d, dtype=np.float32), device="cuda")
    dv = dv / dv.norm()
    for lyr in mod.model.layers:
        for W in (lyr.self_attn.o_proj.weight, lyr.mlp.down_proj.weight):
            W32 = W.data.float()
            W.data.copy_((W32 - lam * torch.outer(dv, dv @ W32)).to(W.dtype))
    E = mod.get_input_embeddings().weight
    TIED = bool(mod.get_output_embeddings() is not None and mod.get_output_embeddings().weight.data_ptr() == E.data_ptr())
    if not TIED:
        E32 = E.data.float()
        E.data.copy_((E32 - lam * torch.outer(E32 @ dv, dv)).to(E.dtype))
    return TIED


CHECK_TEXTS = [p["prompt"] for p in sorted(SETS["extraction"], key=lambda p: hashlib.sha256(p["prompt_id"].encode()).hexdigest())[:50]]


def proj_profile(mod, tok, d, Lsrc):
    """mean |act . d| at the last prompt token, for layers Lsrc..34 on the 50 fixed extraction prompts."""
    layers = list(range(Lsrc, mod.config.num_hidden_layers - 1))
    a = last_token_acts(mod, tok, CHECK_TEXTS, layers)
    dn = np.asarray(d, dtype=np.float32) / np.linalg.norm(d)
    return {L: float(np.abs(a[L] @ dn).mean()) for L in layers}


def build_edited(m, Lsrc, lam, directions, save_to):
    """Load model m, edit with fperp_{m}_L{Lsrc} at strength lam, run the projection check, save to save_to (local VM disk)."""
    mod, tok = hf_load(model_source(m))
    d = directions[f"fperp_{m}_L{Lsrc}"]
    before = proj_profile(mod, tok, d, Lsrc)
    tied = apply_edit(mod, d, lam)
    after = proj_profile(mod, tok, d, Lsrc)
    fall = {L: 1 - after[L] / before[L] for L in before}
    check = {"model": m, "Lsrc": Lsrc, "lam": lam, "tied_embeddings": tied, "embed_edited": not tied,
             "min_fall": float(min(fall.values())), "required": 0.9 * lam, "passed": bool(min(fall.values()) >= 0.9 * lam),
             "fall_by_layer": fall, "before": before, "after": after}
    print(f"  projection check {m} L{Lsrc} lam{lam}: min fall {check['min_fall']:.3f} (need {0.9 * lam:.2f}) -> {'pass' if check['passed'] else 'FAIL (logged)'}")
    import shutil
    if Path(save_to).exists():
        shutil.rmtree(save_to)
    mod.save_pretrained(save_to)
    tok.save_pretrained(save_to)
    del mod
    gc.collect()
    torch.cuda.empty_cache()
    return check


def load_directions(name="s2_directions"):
    d = OUT / name
    assert folder_ok(d), f"{name} not complete"
    z = np.load(d / "directions.npz")
    return {k: z[k] for k in z.files}, json.loads((d / "diagnostics.json").read_text())


def half(seed_id):
    return int(hashlib.sha256(seed_id.encode()).hexdigest(), 16) % 2 == 0


def judge_all(keys, by_key_prompt, by_key_answer, path, gid, util):
    """Judge (prompt, answer) for every key with one guard; resumes from the JSONL checkpoint at path."""
    v = {r["key"]: r for r in load_jsonl(path)}
    todo = [k for k in keys if k not in v]
    print(f"{Path(path).name}: {len(v)} judged, {len(todo)} to go")
    if todo:
        g = load_llm(gid, DTYPE, gpu_memory_utilization=util, max_model_len=8192)
        for i in range(0, len(todo), 800):
            ch = todo[i:i + 800]
            js = judge(g, [by_key_prompt[k] for k in ch], [by_key_answer[k] for k in ch])
            rs = [{"key": k, **j} for k, j in zip(ch, js)]
            append_jsonl(path, rs)
            v.update({r["key"]: r for r in rs})
        free_llm(g)
    return v


def stop_requested():
    return (IN / "STOP").exists() or (OUT / "STOP").exists()


{'extraction': 3864, 'selection': 1528, 'eval_us': 6228, 'eval_jp': 5700}


In [9]:
# XB1: answer the extraction set's harmless twins with ja_s0 (vLLM, greedy, 1024) and judge them with the 0.6B guard (S1 code).
STAGE = OUT / OUTNAME
STAGE.mkdir(parents=True, exist_ok=True)
if folder_ok(STAGE):
    print("already DONE; skipping")
else:
    P = [p for p in SETS["extraction"] if p["kind"] == "twin"]
    by_pid = {p["prompt_id"]: p for p in P}
    ANS = STAGE / "answers_ckpt.jsonl"
    ans = {(r["model"], r["prompt_id"]): r for r in load_jsonl(ANS)}
    for m in MODELS:
        todo = [p for p in P if (m, p["prompt_id"]) not in ans]
        print(m, "to answer:", len(todo))
        if not todo:
            continue
        llm = load_llm(model_source(m), DTYPE, gpu_memory_utilization=0.40)
        for i in range(0, len(todo), 400):
            ch = todo[i:i + 400]
            outs = answer(llm, [p["prompt"] for p in ch])
            rs = [{"model": m, "prompt_id": p["prompt_id"], **o} for p, o in zip(ch, outs)]
            append_jsonl(ANS, rs)
            ans.update({(r["model"], r["prompt_id"]): r for r in rs})
            print(f"  {m}: {i + len(ch)}/{len(todo)}")
        free_llm(llm)
    keys = [f"{m}|{p['prompt_id']}" for m in MODELS for p in P]
    v = judge_all(keys, {k: by_pid[k.split("|")[1]]["prompt"] for k in keys},
                  {k: ans[tuple(k.split("|"))]["answer"] for k in keys}, STAGE / "judge_g06_ckpt.jsonl", GUARD_06, 0.15)
    rows = []
    for k in keys:
        m, pid = k.split("|")
        a, p, j = ans[(m, pid)], by_pid[pid], v[k]
        rows.append({"model": m, "prompt_id": pid, **{f: p[f] for f in ("seed_id", "kind", "severity", "culture", "framing", "lang", "category")},
                     "answer": a["answer"], "tokens": a["tokens"], "finish_reason": a["finish_reason"], "answer_lang": answer_lang(a["answer"]),
                     **{f"g06_{f}": j[f] for f in ("label", "categories", "unsafe", "refused", "raw")}})
    write_jsonl(STAGE / "answers.jsonl", rows)
    df = pd.DataFrame(rows)
    summ = {f"{m}|{l}|{c}": {"twins": int(len(g)), "refused": int((g.g06_refused == 1).sum())}
            for (m, l, c), g in df.groupby(["model", "lang", "culture"])}
    summ_ml = {f"{m}|{l}": f"{int((g.g06_refused == 1).sum())}/{len(g)}" for (m, l), g in df.groupby(["model", "lang"])}
    print("refused harmless twins per model x language:", summ_ml)
    adapter_sha = {f: sha256_file(IN / "adapters" / MODELS[0] / f) for f in ("adapter_config.json", "adapter_model.safetensors")}
    man = publish(STAGE, ["answers.jsonl"], {"stage": OUTNAME, "models": MODELS, "set": "extraction (twins only)", "n_prompts": len(P),
                                             "refused_twins": summ, "refused_twins_model_lang": summ_ml, "adapter_sha256": adapter_sha,
                                             "n_unparsed_g06": int((df.g06_label == "Unparsed").sum()),
                                             "n_truncated": int((df.finish_reason == "length").sum()),
                                             "decoding": {"greedy": True, "max_new_tokens": ANSWER_MAX_NEW_TOKENS, "serving": "vLLM, merged bf16"},
                                             "judge": {"id": GUARD_06, "sha": model_sha(GUARD_06)}, "base": {"id": TINY_AYA, "sha": model_sha(TINY_AYA)}})
    print("published", STAGE)


ja_s0 to answer: 1956


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

... (35 outputs left out)

(MainProcess pid=1926) INFO 11-16 07:12:49 [core_client.py:792] [shutdown] MPClient: start timeout=default


(MainProcess pid=1926) INFO 11-16 07:12:49 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore


refused harmless twins per model x language: {'ja_s0|en': '285/975', 'ja_s0|ja': '258/981'}


published /content/x2/out/s1_labels_ja_s0
